# (05) Jobs: rate-dist curve, amortized P-VAE fits

project = ```iP-VAE```, host = ```yoru```, device = ```any```

**Motivation**: <br>

Create jobs for all the models that'll go on the rate-distortion curve. These are specificall amortized P-VAE models with conv encoder.

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')
gir_dir = os.path.join(git_dir, 'PoissonVAE')

# GitHub
sys.path.insert(0, gir_dir)
from figures.fighelper import *
from main.train_vae import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from analysis.helper import job_runner_script


def divide_list(lst: list, n: int):
	k, m = divmod(len(lst), n)
	lst_divided = [
		lst[
			i * k + min(i, m):
			(i + 1) * k + min(i + 1, m)
		] for i in range(n)
	]
	return lst_divided


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = pjoin(gir_dir, 'scripts')
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

['copyfits.sh', 'fit_vae.sh', 'kill_screens.sh', 'resume_fit.sh', 'run_sessions.sh']

## Betas Amort (yoru)

```<conv+b|lin>```

In [4]:
host = 'yoru'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)

In [5]:
# n_seeds = 5
# seeds = range(1, n_seeds + 1)

model_act_map = {
    'poisson': [None],
    'gaussian': [None, 'relu',' softplus'],
}
betas = [
    0.01, 0.1, 0.4, 0.6,
    0.8, 1.0, 1.2, 1.6,
    2.0, 2.5, 3.0, 4.0,
]

In [6]:
tot = 0

for model_type, latent_act_list in model_act_map.items():
    for latent_act in latent_act_list:
        for beta in betas:
            arg = ' '.join(filter(None, [
                f"--kl_beta {beta}",
                f"--latent_act '{latent_act}'" if latent_act else '',
                f"--comment amort_b-{beta:0.3g}",
            ]))
            gpu_i = tot % torch.cuda.device_count()
            kws = dict(
                device=gpu_i,
                dataset='vH16',
                archi='conv+b|lin',
                model=model_type,
                args=arg,
                seed=1,
            )
            scripts[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

48

In [8]:
scripts = dict(scripts)
print({k: len(v) for k, v in scripts.items()})

{0: 24, 1: 24}

### Save

In [9]:
n_fits = 6

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )
        print(combined.replace('&& ', '&& \n'))

[PROGRESS] 'yoru-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.01 --comment amort_b-0.01 && 
./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.4 --comment amort_b-0.4 && 
./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.8 --comment amort_b-0.8 && 
./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.2 --comment amort_b-1.2

[PROGRESS] 'yoru-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort_b-2 && 
./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 3.0 --comment amort_b-3 && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.01 --comment amort_b-0.01 && 
./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.4 --comment amort_b-0.4

[PROGRESS] 'yoru-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


...

./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.1 --latent_act 'relu' --comment amort_b-0.1 &&
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.6 --latent_act 'relu' --comment amort_b-0.6 &&
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act 'relu' --comment amort_b-1 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --latent_act 'relu' --comment amort_b-1.6

[PROGRESS] 'yoru-cuda1-fit4.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --latent_act 'relu' --comment amort_b-2.5 &&
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0 --latent_act 'relu' --comment amort_b-4 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.1 --latent_act ' softplus' --comment 
amort_b-0.1 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.6 --latent_act ' softplus' --comment 
amort_b-0.6

[PROGRESS] 'yoru-cuda1-fit5.txt' saved at
/home/hadi/Dropbox/git/PoissonVAE/scripts


./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act ' softplus' --comment amort_b-1
&& 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --latent_act ' softplus' --comment 
amort_b-1.6 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --latent_act ' softplus' --comment 
amort_b-2.5 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0 --latent_act ' softplus' --comment amort_b-4

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act ' softplus' --comment amort_b-1
&& 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --latent_act ' softplus' --comment 
amort_b-1.6 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --latent_act ' softplus' --comment 
amort_b-2.5 && 
./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0 --latent_act ' softplus' --comment amort_b-4

In [11]:
print(scripts)

{
    0: [
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.01 --comment amort_b-0.01",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.4 --comment amort_b-0.4",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.8 --comment amort_b-0.8",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.2 --comment amort_b-1.2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort_b-2",
        "./fit_vae.sh '0' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 3.0 --comment amort_b-3",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.01 --comment amort_b-0.01",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.4 --comment amort_b-0.4",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.8 --comment amort_b-0.8",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.2 --comment amort_b-1.2",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --comment amort_b-2",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 3.0 --comment amort_b-3",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.01 --latent_act 'relu' --comment 
amort_b-0.01",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.4 --latent_act 'relu' --comment 
amort_b-0.4",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.8 --latent_act 'relu' --comment 
amort_b-0.8",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.2 --latent_act 'relu' --comment 
amort_b-1.2",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --latent_act 'relu' --comment 
amort_b-2",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 3.0 --latent_act 'relu' --comment 
amort_b-3",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.01 --latent_act ' softplus' --comment
amort_b-0.01",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.4 --latent_act ' softplus' --comment 
amort_b-0.4",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.8 --latent_act ' softplus' --comment 
amort_b-0.8",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.2 --latent_act ' softplus' --comment 
amort_b-1.2",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.0 --latent_act ' softplus' --comment 
amort_b-2",
        "./fit_vae.sh '0' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 3.0 --latent_act ' softplus' --comment 
amort_b-3"
    ],
    1: [
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.1 --comment amort_b-0.1",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.6 --comment amort_b-0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort_b-1",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.6 --comment amort_b-1.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.5 --comment amort_b-2.5",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 4.0 --comment amort_b-4",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.1 --comment amort_b-0.1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.6 --comment amort_b-0.6",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort_b-1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --comment amort_b-1.6",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --comment amort_b-2.5",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0

In [12]:
print(scripts_divided)

[
    [
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.1 --comment amort_b-0.1",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 0.6 --comment amort_b-0.6",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort_b-1",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 1.6 --comment amort_b-1.6"
    ],
    [
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 2.5 --comment amort_b-2.5",
        "./fit_vae.sh '1' 'vH16' 'poisson' 'conv+b|lin' --seed 1 --kl_beta 4.0 --comment amort_b-4",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.1 --comment amort_b-0.1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.6 --comment amort_b-0.6"
    ],
    [
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --comment amort_b-1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --comment amort_b-1.6",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --comment amort_b-2.5",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0 --comment amort_b-4"
    ],
    [
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.1 --latent_act 'relu' --comment 
amort_b-0.1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.6 --latent_act 'relu' --comment 
amort_b-0.6",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act 'relu' --comment 
amort_b-1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --latent_act 'relu' --comment 
amort_b-1.6"
    ],
    [
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --latent_act 'relu' --comment 
amort_b-2.5",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0 --latent_act 'relu' --comment 
amort_b-4",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.1 --latent_act ' softplus' --comment 
amort_b-0.1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 0.6 --latent_act ' softplus' --comment 
amort_b-0.6"
    ],
    [
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.0 --latent_act ' softplus' --comment 
amort_b-1",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 1.6 --latent_act ' softplus' --comment 
amort_b-1.6",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 2.5 --latent_act ' softplus' --comment 
amort_b-2.5",
        "./fit_vae.sh '1' 'vH16' 'gaussian' 'conv+b|lin' --seed 1 --kl_beta 4.0 --latent_act ' softplus' --comment 
amort_b-4"
    ]
]

## Was checking

In [14]:
model_type = 'gaussian'
cfg_vae, cfg_tr = default_configs('vH16', model_type, 'conv+b|lin')

In [15]:
tr = TrainerVAE(
    model=MODEL_CLASSES[model_type](CFG_CLASSES[model_type](**cfg_vae, save=False)),
    cfg=ConfigTrainVAE(**cfg_tr),
    device='cuda:0',
)

In [16]:
from prettytable import PrettyTable

print_num_params(tr.model)

+-------------+------------+
| Module Name | Num Params |
+-------------+------------+
| GaussianVAE |  1.5 Mil   |
|     ———     |    ———     |
|     stem    |    352     |
|     enc     |  1.2 Mil   |
|    fc_enc   |  132.1 K   |
|    fc_dec   |  131.1 K   |
+-------------+------------+